# Complex Fields

Complex-valued `CoefficientFunction`s and `GridFunction`s are automatically detected.
Both real and imaginary parts are uploaded to the GPU in a single interleaved buffer,
enabling efficient visualization of different views (Real, Imag, Abs, Phase) and
phase-sweep animation without any buffer re-upload.

## Scalar Complex Field

A complex grid function is drawn just like a real one. By default, the real part is shown.
A "Complex Mode" dropdown and an "Animate" checkbox appear automatically in the GUI.

In [ ]:
from ngsolve import *
from ngsolve_webgpu import *
from ngsolve_webgpu.cf import CFRenderer
from ngsolve_webgpu.jupyter import Draw

mesh = Mesh(unit_square.GenerateMesh(maxh=0.1))
fes = H1(mesh, order=3, complex=True)
gf = GridFunction(fes)
gf.Set(sin(3*x) + 1j*cos(3*y))

scene = Draw(gf, mesh, order=3)

## Switching Complex Visualization Mode

Use `set_complex_mode` on the `CFRenderer` to switch between views.
Available modes: `"real"`, `"imag"`, `"abs"`, `"phase"`.

This only updates a small uniform on the GPU — no data re-upload.

In [ ]:
for ro in scene.render_objects:
    if isinstance(ro, CFRenderer):
        ro.set_complex_mode("abs")
        break

scene.render()

In [ ]:
for ro in scene.render_objects:
    if isinstance(ro, CFRenderer):
        ro.set_complex_mode("imag")
        break

scene.render()

## Phase Animation

For frequency-domain solutions, the physical field is:

$$u(x, t) = \text{Re}\bigl(u_{\text{complex}}(x) \cdot e^{i\omega t}\bigr)$$

Call `animate_phase(scene)` to start a smooth phase sweep.
Each frame costs only an 8-byte uniform write — the Bernstein coefficient buffers stay untouched.

You can also use the "Animate" checkbox in the GUI.

In [ ]:
for ro in scene.render_objects:
    if isinstance(ro, CFRenderer):
        ro.animate_phase(scene, speed=0.2)  # 1 full cycle per second
        break

In [ ]:
# Stop the animation
for ro in scene.render_objects:
    if isinstance(ro, CFRenderer):
        ro.stop_animation()
        break

## Complex Vector Field

Vector-valued complex fields work the same way. Component selection and complex mode are independent.

In [ ]:
fes_vec = VectorH1(mesh, order=3, complex=True)
gf_vec = GridFunction(fes_vec)
gf_vec.Set(CF((x + 1j*y, 2*x - 1j*y)))

scene2 = Draw(gf_vec, mesh, order=3)

In [ ]:
# Show imaginary part of component 1
for ro in scene2.render_objects:
    if isinstance(ro, CFRenderer):
        ro.set_component(1)
        ro.set_complex_mode("real")
        break

scene2.render()